In [1]:
import pandas as pd
import numpy as np
import pickle
from sklearn.ensemble import IsolationForest

In [2]:
import pandas as pd
import numpy as np
import pickle
from sklearn.ensemble import IsolationForest

In [3]:
X_train = pd.read_csv('../data/X_train.csv')
y_train = pd.read_csv('../data/y_train.csv').squeeze()
X_test  = pd.read_csv('../data/X_test.csv')
y_test  = pd.read_csv('../data/y_test.csv').squeeze()


In [4]:
X_train_benign = X_train[y_train == 0]

In [5]:
iso_forest = IsolationForest(
    contamination=0.05,
    random_state=42,
    n_jobs=-1
)


In [6]:
iso_forest.fit(X_train_benign)
pickle.dump(iso_forest, open('../results/isolation_forest_model.pkl', 'wb'))
print(f"Trained on {X_train_benign.shape[0]} benign flows.")

Trained on 351746 benign flows.


In [7]:
test_preds   = iso_forest.predict(X_test)
test_scores  = iso_forest.decision_function(X_test)

# Convert: -1 → 1 (attack), 1 → 0 (benign)
test_preds_binary = (test_preds == -1).astype(int)

from sklearn.metrics import classification_report
print(classification_report(y_test, test_preds_binary))

# Save scores for later fusion
pd.DataFrame({'anomaly_score': test_scores}).to_csv(
    '../results/test_anomaly_scores.csv', index=False
)
print("Anomaly scores saved.")

              precision    recall  f1-score   support

           0       0.81      0.95      0.87     87937
           1       0.87      0.60      0.71     50345

    accuracy                           0.82    138282
   macro avg       0.84      0.78      0.79    138282
weighted avg       0.83      0.82      0.81    138282

Anomaly scores saved.
